# DSPy Muay Thai Multiple Combos Classifier

This notebook implements a DSPy-based system using BootstrapFewShot optimizer to help Google Gemini classify multiple Muay Thai strike combos in videos. It evaluates the model's ability to identify separate combos and recognize when one combo ends and another begins.


## Section 1: Setup and Configuration


In [1]:
import dspy
import google.generativeai as genai
import json
from pathlib import Path
import os
from dotenv import load_dotenv
import random
from typing import List, Dict, Tuple
import re

# Load environment variables
load_dotenv()

# Get Gemini API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in environment variables. Please set it in .env file.")

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)

# Configure DSPy with Gemini 2.5 Flash
try:
    lm = dspy.LM(model="gemini/gemini-2.5-flash", api_key=GEMINI_API_KEY)
    print('using gemini-2.5-flash')
except:
    print("Error: Failed to configure Gemini 2.5 Flash. Please check your API key and model availability.")

dspy.configure(lm=lm)

# Data paths
DATA_ROOT = Path("data/multiple_combos")
SHADOW_DIR = DATA_ROOT / "shadow"
HEAVY_BAG_DIR = DATA_ROOT / "heavy_bag"

# Training split sizes (3 shadow + 4 heavy_bag = 7 total for training)
SHADOW_TRAIN_SIZE = 3
HEAVY_BAG_TRAIN_SIZE = 4

print("Setup complete!")
print(f"Data directories: {SHADOW_DIR}, {HEAVY_BAG_DIR}")


using gemini-2.5-flash
Setup complete!
Data directories: data\multiple_combos\shadow, data\multiple_combos\heavy_bag


## Section 2: Data Loading


In [2]:
def load_labels(json_path: Path) -> Dict[str, List[List[str]]]:
    """Load true labels from JSON file. For multiple_combos, each video has multiple combos."""
    with open(json_path, 'r') as f:
        return json.load(f)

def create_examples(labels: Dict[str, List[List[str]]], video_dir: Path, dataset_name: str) -> List[dspy.Example]:
    """Create DSPy examples from labels and video directory.
    
    For multiple_combos, each video has multiple combos (list of lists).
    We keep the combos structure as arrays of arrays for evaluation.
    """
    examples = []
    for video_id, combos in labels.items():
        video_path = video_dir / f"{video_id}.mov"
        if video_path.exists():
            example = dspy.Example(
                video_path=str(video_path),
                combos=combos  # Keep as array of arrays
            ).with_inputs("video_path")
            examples.append(example)
        else:
            print(f"Warning: Video {video_path} not found")
    return examples

# Load labels from both datasets
shadow_labels = load_labels(SHADOW_DIR / "true_labels.json")
heavy_bag_labels = load_labels(HEAVY_BAG_DIR / "true_labels.json")

# Create examples
shadow_examples = create_examples(shadow_labels, SHADOW_DIR, "shadow")
heavy_bag_examples = create_examples(heavy_bag_labels, HEAVY_BAG_DIR, "heavy_bag")

print(f"Loaded {len(shadow_examples)} shadow examples")
print(f"Loaded {len(heavy_bag_examples)} heavy_bag examples")


Loaded 12 shadow examples
Loaded 19 heavy_bag examples


In [3]:
# Split data into train and validation
# Use the same random seed (42) as other notebooks for reproducibility
# Training set: 3 shadow + 4 heavy_bag = 7 total

random.seed(42)  # For reproducibility

# Shadow split (3 for training)
shadow_indices = list(range(len(shadow_examples)))
random.shuffle(shadow_indices)
shadow_train_indices = set(shadow_indices[:SHADOW_TRAIN_SIZE])
shadow_train = [shadow_examples[i] for i in shadow_train_indices]
shadow_val = [shadow_examples[i] for i in range(len(shadow_examples)) if i not in shadow_train_indices]

# Heavy bag split (4 for training)
heavy_bag_indices = list(range(len(heavy_bag_examples)))
random.shuffle(heavy_bag_indices)
heavy_bag_train_indices = set(heavy_bag_indices[:HEAVY_BAG_TRAIN_SIZE])
heavy_bag_train = [heavy_bag_examples[i] for i in heavy_bag_train_indices]
heavy_bag_val = [heavy_bag_examples[i] for i in range(len(heavy_bag_examples)) if i not in heavy_bag_train_indices]

# Combine train and validation sets
train_examples = shadow_train + heavy_bag_train
val_examples = shadow_val + heavy_bag_val

print(f"Training set: {len(train_examples)} examples ({len(shadow_train)} shadow + {len(heavy_bag_train)} heavy_bag)")
print(f"Validation set: {len(val_examples)} examples ({len(shadow_val)} shadow + {len(heavy_bag_val)} heavy_bag)")

# Display some statistics
all_strikes = set()
total_combos = 0
for ex in train_examples + val_examples:
    for combo in ex.combos:
        all_strikes.update(combo)
        total_combos += 1

print(f"\nUnique strike types: {sorted(all_strikes)}")
print(f"Total unique strike types: {len(all_strikes)}")
print(f"Total combos across all videos: {total_combos}")
print(f"Average combos per video: {total_combos / len(train_examples + val_examples):.2f}")


Training set: 7 examples (3 shadow + 4 heavy_bag)
Validation set: 24 examples (9 shadow + 15 heavy_bag)

Unique strike types: ['cross', 'jab', 'knee', 'lead_elbow', 'lead_hook', 'lead_roundhouse', 'lead_uppercut', 'low_kick', 'rear_elbow', 'rear_hook', 'rear_uppercut', 'roundhouse', 'spinning_elbow', 'switch_kick', 'switch_knee', 'teep']
Total unique strike types: 16
Total combos across all videos: 93
Average combos per video: 3.00


## Section 3: DSPy Signature and Module


In [4]:
class MultipleCombosClassifier(dspy.Signature):
    """Classify Muay Thai strikes in a video sequence with multiple combos. Watch the video and identify all the separate combos in the order they appear. A combo is a sequence of strikes performed together. When the fighter pauses or resets, a new combo begins. Return a JSON array of arrays, where each inner array represents one combo."""
    video_path: str = dspy.InputField(desc="Path to video file")
    combos: str = dspy.OutputField(desc="JSON array of arrays, where each inner array represents one combo, e.g. [[\"jab\", \"cross\", \"lead_hook\"], [\"jab\", \"cross\", \"roundhouse\"], [\"teep\"]]")


In [5]:
class MuayThaiMultipleCombosClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classifier = dspy.ChainOfThought(MultipleCombosClassifier)
    
    def forward(self, video_path: str):
        return self.classifier(video_path=video_path)

# Initialize classifier
classifier = MuayThaiMultipleCombosClassifier()

print("Classifier module created!")


Classifier module created!


## Section 4: BootstrapFewShot Optimizer


In [6]:
# Helper function to parse combos from DSPy prediction
def parse_combos_from_prediction(pred) -> List[List[str]]:
    """Parse combos from DSPy prediction, handling various formats."""
    try:
        if isinstance(pred.combos, str):
            combos = json.loads(pred.combos)
        else:
            combos = pred.combos
    except (json.JSONDecodeError, AttributeError):
        combos = []
        if hasattr(pred, 'combos'):
            pred_str = str(pred.combos)
            # Try to find nested arrays: [[...], [...]]
            json_match = re.search(r'\[(\[.*?\](?:,\s*\[.*?\])*)\]', pred_str, re.DOTALL)
            if json_match:
                try:
                    combos = json.loads(json_match.group())
                except:
                    pass
            # If that fails, try single array
            if not combos:
                match = re.search(r'\[.*?\]', pred_str)
                if match:
                    try:
                        strikes = json.loads(match.group())
                        if isinstance(strikes, list) and all(isinstance(s, str) for s in strikes):
                            combos = [strikes]  # Wrap as single combo
                    except:
                        pass
    
    # Validate it's an array of arrays
    if isinstance(combos, list):
        if all(isinstance(combo, list) for combo in combos):
            return combos
        elif all(isinstance(strike, str) for strike in combos):
            return [combos]  # Wrap flat array as single combo
    
    return []

# Evaluation metrics for arrays of arrays (multiple combos) - adapted for DSPy
def combo_boundary_accuracy_metric(gold, pred, trace=None):
    """Calculate accuracy of combo boundaries (F1 score for boundary detection)."""
    true_combos = gold.combos if hasattr(gold, 'combos') else gold
    predicted_combos = parse_combos_from_prediction(pred)
    
    if not isinstance(true_combos, list) or not all(isinstance(c, list) for c in true_combos):
        return 0.0
    
    # Flatten true combos with boundary markers
    true_flat = []
    true_boundaries = set()
    pos = 0
    for combo in true_combos:
        if combo:  # Only add boundary if combo is non-empty
            true_boundaries.add(pos)
            true_flat.extend(combo)
            pos += len(combo)
    
    # Flatten predicted combos with boundary markers
    pred_flat = []
    pred_boundaries = set()
    pos = 0
    for combo in predicted_combos:
        if combo:  # Only add boundary if combo is non-empty
            pred_boundaries.add(pos)
            pred_flat.extend(combo)
            pos += len(combo)
    
    if not true_boundaries:
        return 1.0 if not pred_boundaries else 0.0
    
    # Calculate boundary precision and recall
    correct_boundaries = len(true_boundaries & pred_boundaries)
    precision = correct_boundaries / len(pred_boundaries) if pred_boundaries else 0.0
    recall = correct_boundaries / len(true_boundaries) if true_boundaries else 0.0
    
    # F1 score for boundaries
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def exact_match_metric(gold, pred, trace=None):
    """Evaluate exact match between predicted and true combos (arrays of arrays)."""
    true_combos = gold.combos if hasattr(gold, 'combos') else gold
    predicted_combos = parse_combos_from_prediction(pred)
    
    if not isinstance(true_combos, list) or not all(isinstance(c, list) for c in true_combos):
        return False
    
    return true_combos == predicted_combos

def combo_wise_accuracy_metric(gold, pred, trace=None):
    """Calculate accuracy of individual combos (how many combos match exactly)."""
    true_combos = gold.combos if hasattr(gold, 'combos') else gold
    predicted_combos = parse_combos_from_prediction(pred)
    
    if not isinstance(true_combos, list) or not all(isinstance(c, list) for c in true_combos):
        return 0.0
    
    if not true_combos:
        return 1.0 if not predicted_combos else 0.0
    if not predicted_combos:
        return 0.0
    
    # Match combos (order matters)
    min_len = min(len(true_combos), len(predicted_combos))
    exact_matches = sum(1 for i in range(min_len) if true_combos[i] == predicted_combos[i])
    
    return exact_matches / len(true_combos)

print("Evaluation metrics defined!")


Evaluation metrics defined!


In [7]:
# Configure BootstrapFewShot optimizer
from dspy.teleprompt import BootstrapFewShot

# Use combo_boundary_accuracy as primary metric since it tests the key capability
optimizer = BootstrapFewShot(
    metric=combo_boundary_accuracy_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=8
)

print("Optimizer configured!")
print(f"Primary metric: combo_boundary_accuracy (F1 score for boundary detection)")
print(f"Max bootstrapped demos: 4")
print(f"Max labeled demos: 8")


Optimizer configured!
Primary metric: combo_boundary_accuracy (F1 score for boundary detection)
Max bootstrapped demos: 4
Max labeled demos: 8


In [8]:
# Run optimization on training set
# This may take a while as it needs to process videos through Gemini API

print("Starting optimization...")
print(f"Training on {len(train_examples)} examples")

try:
    optimized_classifier = optimizer.compile(
        student=classifier,
        trainset=train_examples
    )
    print("\nOptimization complete!")
except Exception as e:
    print(f"Error during optimization: {e}")
    print("This may be due to API rate limits or video processing issues.")
    print("You may need to adjust the optimizer parameters or reduce the training set size.")
    raise


Starting optimization...
Training on 7 examples


 57%|█████▋    | 4/7 [00:00<00:00,  4.36it/s]

Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.

Optimization complete!


## Section 5: Evaluation


In [9]:
# Additional evaluation metrics for comprehensive analysis
def combo_count_accuracy(true_combos: List[List[str]], predicted_combos: List[List[str]]) -> float:
    """Calculate accuracy of combo count (how many combos detected vs actual)."""
    if not true_combos:
        return 1.0 if not predicted_combos else 0.0
    if not predicted_combos:
        return 0.0
    
    # Penalize based on difference in count
    count_diff = abs(len(true_combos) - len(predicted_combos))
    max_count = max(len(true_combos), len(predicted_combos))
    return 1.0 - (count_diff / max_count) if max_count > 0 else 1.0

def combo_sequence_accuracy(true_combos: List[List[str]], predicted_combos: List[List[str]]) -> float:
    """Calculate sequence accuracy across all combos (positional accuracy of all strikes)."""
    # Flatten both
    true_flat = []
    for combo in true_combos:
        true_flat.extend(combo)
    
    pred_flat = []
    for combo in predicted_combos:
        pred_flat.extend(combo)
    
    if not pred_flat or not true_flat:
        return 0.0
    
    # Calculate positional accuracy
    min_len = min(len(pred_flat), len(true_flat))
    if min_len == 0:
        return 0.0
    
    matches = sum(1 for i in range(min_len) if pred_flat[i] == true_flat[i])
    return matches / len(true_flat)

# Evaluate optimized model on validation set
print(f"Evaluating on {len(val_examples)} validation examples...")

results = []
exact_matches = 0
total_accuracy = 0.0
total_combo_count_acc = 0.0
total_boundary_acc = 0.0
total_combo_wise_acc = 0.0

for i, example in enumerate(val_examples):
    try:
        prediction = optimized_classifier(video_path=example.video_path)
        
        # Parse prediction
        predicted_combos = parse_combos_from_prediction(prediction)
        true_combos = example.combos
        
        # Calculate all metrics
        exact_match = exact_match_metric(example, prediction)
        combo_count_acc = combo_count_accuracy(true_combos, predicted_combos)
        boundary_acc = combo_boundary_accuracy_metric(example, prediction)
        sequence_acc = combo_sequence_accuracy(true_combos, predicted_combos)
        combo_wise_acc = combo_wise_accuracy_metric(example, prediction)
        
        if exact_match:
            exact_matches += 1
        total_accuracy += sequence_acc
        total_combo_count_acc += combo_count_acc
        total_boundary_acc += boundary_acc
        total_combo_wise_acc += combo_wise_acc
        
        results.append({
            'video_path': example.video_path,
            'true_combos': true_combos,
            'predicted_combos': predicted_combos,
            'exact_match': exact_match,
            'combo_count_accuracy': combo_count_acc,
            'combo_boundary_accuracy': boundary_acc,
            'sequence_accuracy': sequence_acc,
            'combo_wise_accuracy': combo_wise_acc
        })
        
        if (i + 1) % 5 == 0:
            print(f"Processed {i + 1}/{len(val_examples)} examples...")
            
    except Exception as e:
        print(f"Error processing {example.video_path}: {e}")
        results.append({
            'video_path': example.video_path,
            'true_combos': example.combos,
            'predicted_combos': [],
            'exact_match': False,
            'combo_count_accuracy': 0.0,
            'combo_boundary_accuracy': 0.0,
            'sequence_accuracy': 0.0,
            'combo_wise_accuracy': 0.0,
            'error': str(e)
        })

print("\nEvaluation complete!")


Evaluating on 24 validation examples...
Processed 5/24 examples...
Processed 10/24 examples...
Processed 15/24 examples...
Processed 20/24 examples...

Evaluation complete!


In [10]:
# Calculate and display metrics
exact_match_accuracy = exact_matches / len(val_examples) if val_examples else 0
average_sequence_accuracy = total_accuracy / len(val_examples) if val_examples else 0
avg_combo_count_acc = total_combo_count_acc / len(results) if results else 0.0
avg_boundary_acc = total_boundary_acc / len(results) if results else 0.0
avg_combo_wise_acc = total_combo_wise_acc / len(results) if results else 0.0

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Exact Match Accuracy: {exact_match_accuracy:.2%} ({exact_matches}/{len(val_examples)})")
print(f"Average Sequence Accuracy: {average_sequence_accuracy:.2%}")
print(f"Average Combo Count Accuracy: {avg_combo_count_acc:.2%}")
print(f"Average Boundary Accuracy: {avg_boundary_acc:.2%}")
print(f"Average Combo-wise Accuracy: {avg_combo_wise_acc:.2%}")
print()

# Display some example predictions
print("Sample Predictions:")
print("-" * 60)
for i, result in enumerate(results[:5], 1):
    print(f"\nExample {i}:")
    print(f"  Video: {Path(result['video_path']).name}")
    print(f"  True:  {result['true_combos']}")
    print(f"  Pred:  {result['predicted_combos']}")
    print(f"  Exact Match: {'✓' if result['exact_match'] else '✗'}")
    print(f"  Boundary Acc: {result.get('combo_boundary_accuracy', 0.0):.2%}")
    print(f"  Sequence Acc: {result.get('sequence_accuracy', 0.0):.2%}")


EVALUATION RESULTS
Exact Match Accuracy: 0.00% (0/24)
Average Sequence Accuracy: 20.16%
Average Combo Count Accuracy: 77.64%
Average Boundary Accuracy: 42.36%
Average Combo-wise Accuracy: 0.00%

Sample Predictions:
------------------------------------------------------------

Example 1:
  Video: 1.mov
  True:  [['jab'], ['jab', 'cross', 'lead_hook'], ['jab', 'cross', 'lead_hook', 'lead_hook'], ['jab', 'lead_uppercut', 'cross', 'lead_hook']]
  Pred:  [['jab', 'cross', 'lead_hook', 'low_kick'], ['jab', 'cross', 'roundhouse'], ['lead_uppercut', 'cross', 'low_kick']]
  Exact Match: ✗
  Boundary Acc: 57.14%
  Sequence Acc: 25.00%

Example 2:
  Video: 2.mov
  True:  [['cross', 'lead_hook', 'cross'], ['jab', 'cross', 'lead_uppercut', 'roundhouse'], ['jab', 'lead_elbow', 'roundhouse']]
  Pred:  [['jab', 'cross', 'lead_hook', 'roundhouse'], ['jab', 'cross', 'low_kick'], ['jab', 'cross', 'lead_hook', 'teep'], ['jab', 'cross', 'roundhouse']]
  Exact Match: ✗
  Boundary Acc: 57.14%
  Sequence Acc:

In [11]:
# Calculate per-strike accuracy (flattened across all combos)
from collections import defaultdict

strike_stats = defaultdict(lambda: {'correct': 0, 'total': 0, 'predicted': 0})

for result in results:
    # Flatten true combos
    true_strikes = []
    for combo in result['true_combos']:
        true_strikes.extend(combo)
    
    # Flatten predicted combos
    pred_strikes = []
    for combo in result['predicted_combos']:
        pred_strikes.extend(combo)
    
    # Count true strikes
    for strike in true_strikes:
        strike_stats[strike]['total'] += 1
    
    # Count predicted strikes
    for strike in pred_strikes:
        strike_stats[strike]['predicted'] += 1
    
    # Count correct predictions at each position
    min_len = min(len(true_strikes), len(pred_strikes))
    for i in range(min_len):
        if true_strikes[i] == pred_strikes[i]:
            strike_stats[true_strikes[i]]['correct'] += 1

print("\nPer-Strike Accuracy (across all combos):")
print("-" * 60)
for strike in sorted(strike_stats.keys()):
    stats = strike_stats[strike]
    accuracy = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"  {strike:20s}: {accuracy:6.2%} ({stats['correct']}/{stats['total']} correct, {stats['predicted']} predicted)")



Per-Strike Accuracy (across all combos):
------------------------------------------------------------
  cross               : 28.85% (15/52 correct, 78 predicted)
  jab                 : 37.10% (23/62 correct, 75 predicted)
  knee                :  0.00% (0/2 correct, 0 predicted)
  lead_elbow          :  0.00% (0/13 correct, 0 predicted)
  lead_hook           : 17.65% (6/34 correct, 37 predicted)
  lead_roundhouse     :  0.00% (0/2 correct, 0 predicted)
  lead_uppercut       :  0.00% (0/14 correct, 10 predicted)
  low_kick            : 33.33% (1/3 correct, 28 predicted)
  rear_elbow          :  4.55% (1/22 correct, 4 predicted)
  rear_hook           :  0.00% (0/2 correct, 0 predicted)
  rear_uppercut       :  0.00% (0/1 correct, 1 predicted)
  roundhouse          :  0.00% (0/19 correct, 40 predicted)
  spinning_elbow      :  0.00% (0/4 correct, 0 predicted)
  switch_kick         :  0.00% (0/0 correct, 1 predicted)
  switch_knee         :  0.00% (0/3 correct, 0 predicted)
  teep      

## Section 6: Results and Analysis


In [12]:
# Save optimized prompts and results
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# Save evaluation results
results_data = {
    'metrics': {
        'exact_match_accuracy': exact_match_accuracy,
        'average_sequence_accuracy': average_sequence_accuracy,
        'average_combo_count_accuracy': avg_combo_count_acc,
        'average_boundary_accuracy': avg_boundary_acc,
        'average_combo_wise_accuracy': avg_combo_wise_acc,
        'total_examples': len(val_examples),
        'exact_matches': exact_matches
    },
    'results': results,
    'strike_stats': {k: dict(v) for k, v in strike_stats.items()}
}

with open(output_dir / "multiple_combos_dspy.json", 'w') as f:
    json.dump(results_data, f, indent=2)

print(f"Results saved to {output_dir / 'multiple_combos_dspy.json'}")


Results saved to results\multiple_combos_dspy.json


In [13]:
# Display summary
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Model: Gemini 2.5 Flash")
print(f"Optimizer: BootstrapFewShot")
print(f"Dataset: Multiple Combos (videos with multiple combos per video)")
print(f"Training examples: {len(train_examples)} ({len(shadow_train)} shadow + {len(heavy_bag_train)} heavy_bag)")
print(f"Validation examples: {len(val_examples)} ({len(shadow_val)} shadow + {len(heavy_bag_val)} heavy_bag)")
print(f"\nFinal Performance:")
print(f"  Exact Match Accuracy: {exact_match_accuracy:.2%}")
print(f"  Average Sequence Accuracy: {average_sequence_accuracy:.2%}")
print(f"  Average Combo Count Accuracy: {avg_combo_count_acc:.2%}")
print(f"  Average Boundary Accuracy: {avg_boundary_acc:.2%}")
print(f"  Average Combo-wise Accuracy: {avg_combo_wise_acc:.2%}")
print(f"\nResults saved to: {output_dir}/multiple_combos_dspy.json")
print("\nNote: This evaluation tests the model's ability to recognize combo boundaries.")
print("Compare these results with baseline and agent results for evaluation.")
print("=" * 60)



SUMMARY
Model: Gemini 2.5 Flash
Optimizer: BootstrapFewShot
Dataset: Multiple Combos (videos with multiple combos per video)
Training examples: 7 (3 shadow + 4 heavy_bag)
Validation examples: 24 (9 shadow + 15 heavy_bag)

Final Performance:
  Exact Match Accuracy: 0.00%
  Average Sequence Accuracy: 20.16%
  Average Combo Count Accuracy: 77.64%
  Average Boundary Accuracy: 42.36%
  Average Combo-wise Accuracy: 0.00%

Results saved to: results/multiple_combos_dspy.json

Note: This evaluation tests the model's ability to recognize combo boundaries.
Compare these results with baseline and agent results for evaluation.
